# T-E2：AI Agent 自动生成股市周报 — 数据清洗

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI Agent 自动生成股市周报 |
| 小组   | Team02 |
| 成员   | 王佳（25210244）、刘雄（25210196）、黎㵆筠（25210155）、郑嘉豪（25210307）、刘子瑜（25210198）、陈春洁（25210115）、王占溪（25210249）、黎沛鑫（25210156） |
| GitHub | https://github.com/lizi586586-code/T-E2-AI-Agent-Automated-Stock-Market-Weekly-Report |
| Pages  | https://htmlpreview.github.io/?https://github.com/lizi586586-code/T-E2-AI-Agent-Automated-Stock-Market-Weekly-Report/blob/main/output/weekly_report.html |
| 日期   | 2026-05-18 |


## 任务说明

本步骤读取 `data_raw/` 中的原始数据，执行标准数据清洗流程：缺失值检查与处理 → 去重 → 列名中文化 → 日期格式统一 → 数值精度统一。清洗后的数据保存至 `data_clean/`，供后续分析使用。

In [ ]:
import pandas as pd
import numpy as np

# 读取原始数据
df_a = pd.read_csv("data_raw/A股原始数据.csv", encoding="utf-8-sig")
df_us = pd.read_csv("data_raw/美股原始数据.csv", encoding="utf-8-sig")

print(f"A股数据: {df_a.shape[0]} 行 × {df_a.shape[1]} 列")
print(f"美股数据: {df_us.shape[0]} 行 × {df_us.shape[1]} 列")

In [ ]:
# ============================================
# 1. 查看缺失值
# ============================================

print("=" * 40)
print("A股缺失值统计")
print("=" * 40)
print(df_a.isnull().sum())

print("\n" + "=" * 40)
print("美股缺失值统计")
print("=" * 40)
print(df_us.isnull().sum())

### 缺失值检查结果

原始数据无明显缺失值。金融时序数据偶尔会出现停牌导致的空值，但本周各指数均正常交易。

In [ ]:
# ============================================
# 2. 处理缺失值
# ============================================

# 删除全部为空的行
df_a.dropna(how="all", inplace=True)
df_us.dropna(how="all", inplace=True)

# 数值列用前一日数据填充（金融时序数据用前值填充最合理）
num_cols_a = df_a.select_dtypes(include=[np.number]).columns
num_cols_us = df_us.select_dtypes(include=[np.number]).columns
df_a[num_cols_a] = df_a[num_cols_a].fillna(method="ffill")
df_us[num_cols_us] = df_us[num_cols_us].fillna(method="ffill")

# 若仍有缺失（首行无前值），用 0 填充
df_a.fillna(0, inplace=True)
df_us.fillna(0, inplace=True)

print("缺失值处理完成")
print(f"A股剩余缺失: {df_a.isnull().sum().sum()}")
print(f"美股剩余缺失: {df_us.isnull().sum().sum()}")

### 缺失值处理

采用前向填充法（ffill）处理可能存在的缺失值——用前一交易日数据填补，保持趋势连续性。若首行无前值可填，则以 0 填充。该策略适用于金融时序数据的特性。

In [ ]:
# ============================================
# 3. 去重
# ============================================

before_a = len(df_a)
before_us = len(df_us)

df_a.drop_duplicates(inplace=True)
df_us.drop_duplicates(inplace=True)

print(f"A股去重: {before_a} → {len(df_a)} (删除 {before_a - len(df_a)} 条)")
print(f"美股去重: {before_us} → {len(df_us)} (删除 {before_us - len(df_us)} 条)")

### 去重结果

本周数据无重复记录。如数据源出现重复推送，`drop_duplicates()` 可有效去重。

In [ ]:
# ============================================
# 4. 统一列名与数据格式
# ============================================

# 统一列名（中文命名，方便阅读）
col_map = {
    "date": "日期",
    "open": "开盘价",
    "high": "最高价",
    "low": "最低价",
    "close": "收盘价",
    "volume": "成交量",
    "index_name": "指数名称",
}
df_a.rename(columns=col_map, inplace=True)
df_us.rename(columns=col_map, inplace=True)

# 日期列统一为 datetime 格式
for df in [df_a, df_us]:
    date_col = "日期" if "日期" in df.columns else "date"
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# 数值列统一保留 2 位小数
for df in [df_a, df_us]:
    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        df[c] = df[c].round(2)

# 按日期排序
df_a.sort_values("日期", inplace=True)
df_us.sort_values("日期", inplace=True)

print("列名与格式统一完成")
print(f"A股列名: {list(df_a.columns)}")
print(f"美股列名: {list(df_us.columns)}")

### 列名与格式统一

- 列名英文转中文（date→日期, open→开盘价 等），提升可读性
- 日期列统一为 `datetime` 格式
- 数值列保留 2 位小数，与金融数据惯例一致
- 按日期升序排列，确保时序分析正确

In [ ]:
# ============================================
# 5. 保存清洗后数据到 data_clean
# ============================================

df_a.to_csv("data_clean/A股清洗后数据.csv", index=False, encoding="utf-8-sig")
print("A股清洗后数据保存完成 → data_clean/A股清洗后数据.csv")

df_us.to_csv("data_clean/美股清洗后数据.csv", index=False, encoding="utf-8-sig")
print("美股清洗后数据保存完成 → data_clean/美股清洗后数据.csv")

### 数据保存

清洗后数据保存至 `data_clean/`，A 股和美股分别存储。

In [ ]:
# ============================================
# 清洗结果概览
# ============================================

print("=" * 50)
print("A股清洗后数据概览")
print("=" * 50)
print(f"行数: {df_a.shape[0]}, 列数: {df_a.shape[1]}")
display(df_a.head())

print("\n" + "=" * 50)
print("美股清洗后数据概览")
print("=" * 50)
print(f"行数: {df_us.shape[0]}, 列数: {df_us.shape[1]}")
display(df_us.head())

## 结果解读

清洗流程完整执行，未发现数据质量问题。本周 A 股和美股指数数据完整性良好，无需额外人工干预。清洗后的数据已按标准格式整理，可直接进入实证分析阶段。